In [2]:
pip install chromadb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


- *PyMuPDF:* 
    - Python Library to read and manipulate PDF files. 
    - It does text and image extraction in one library and is much faster on large docs than pypdf/pdfplumber.

- *sentence-transformers:* 
    - Turns text (and, with the right model, images) into embedding vectors —> fixed-length arrays of floats where similar meanings land close together. This is the piece that makes semantic search possible.
    - Example:
        - Text Model:
            ```
            model = SentenceTransformer("all-MiniLM-L6-v2")
            ```
        - Text + Images together ->  Use a CLIP(Contrastive Language-Image Pre-training) multimodal AI model or any other multimodal model:
            ```
            SentenceTransformer("clip-ViT-B-32") which embeds both images and text into the same vector space, so a text query can retrieve a relevant image. 
            ```

- *pillow:*
    - standard Python image library to open, convert, resize, crop, save images.

In [7]:
pip install pymupdf sentence-transformers chromadb pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
from typing import List, Dict, Any # used for type hinting in Python, allowing you to specify the expected types of variables, function parameters, and return values.

import fitz # used for importing the PyMuPDF library
from PIL import Image
import numpy as np
from sentence_transformers import SentenceTransformer # gonna help me out in downloading a model from Hugging Face and then using it.
import chromadb

c:\Users\AdityaRajPanda\OneDrive - Cubastion Consulting Pvt Ltd\Desktop\genai-bootcamp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path

BASE_DIR = Path.cwd()                     # the 03_VectorDBs/ folder when run from VS Code / Jupyter
PDF_PATH = BASE_DIR / "attention_is_all_you_need.pdf"
FIGURES_DIR = BASE_DIR / "figures" # where the extracted images will be stored

FIGURES_DIR.mkdir(exist_ok=True)

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"Put attention_is_all_you_need.pdf next to this notebook ({BASE_DIR})"
    )


In [3]:
import os
import chromadb
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))   # .env sits in the repo root, one level above this notebook

client = chromadb.CloudClient(
    api_key=os.environ["CHROMA_DB_API_KEY"],
    tenant=os.environ["CHROMA_DB_TENANT"],
    database=os.environ["CHROMA_DB_DATABASE"],
)

- The Simple strategy is to create two separate *"collections"* in my chromadb database(we have manually created the database).
    - one for textual data
    - one for images

In [4]:
FIG_COLLECTION_NAME = "attention_figures" 
TEXT_COLLECTION_NAME = "attention_text_chunks" 

In [5]:
# We are using the "clip-ViT-B-32" model. The below code is taken from Hugging Face.

from sentence_transformers import SentenceTransformer, util # util is helper module that provides useful functions for common tasks like computing similarity scores between sentences or clustering texts

# Load CLIP model
clip_model = SentenceTransformer('clip-ViT-B-32')

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 819.14it/s]


- The output embedding dimension of the clip-ViT-B-32 model is **512** which simply means the model produces 512-dimensional vectors.
- 512 defines the size of the representation space the model uses.
- *Dimension* here means the length (size) of the embedding vector that the model outputs.
- In models like CLIP, both images and text are converted into fixed-length numerical vectors(embeddings) that live in exactly the same mathematical space.
- Because they are in the same space, you can directly compare any image vector with any text vector using simple geometric measures such as cosine similarity or Euclidean distance.
- Embedding dimension = 512 means each output vector consists of exactly 512 numbers (usually floating-point values).
- So an embedding looks conceptually like:
```
[0.12, -0.45, 0.78, … , 0.03] ← 512 values in total
```
- Higher dimensions can in principle capture more nuanced information, but they also cost more memory and computation.

In [6]:
# code for converting textual data into embeddings

def embed_text(text: str) -> np.ndarray:
    """
    Convert a given text into a vector embedding using the CLIP model.
    """
    # convert_to_numpy=True explicitly forces the method to return the final embeddings as a NumPy array
    # show_progress_bar is used to enable a progress bar in your interface or code. You can set it 'True' or 'False
    text_emb = clip_model.encode([text], convert_to_numpy=True, show_progress_bar=False)
    return text_emb[0] # extracts that first row from 2D array

In [7]:
# code for converting image data into embeddings

def embed_image(path: str) -> np.ndarray:
    """
    Convert an image at the given path into a vector embedding using the CLIP model.
    """
    img = Image.open(path).convert("RGB")  # Open the image and convert it to RGB format
    img_emb = clip_model.encode([img], convert_to_numpy=True, show_progress_bar=True)
    return img_emb[0]

### Understanding 1D and 2D Arrays

#### 1D Array (One-Dimensional)
- **Structure:** A single row of values (like a line)
- **Shape:** `(n,)` where n = number of elements
- **Example:**
```python
  array_1d = [0.12, 0.56, 0.90, 0.34, 0.78]
  shape: (5,)  # 5 elements in ONE row
```
- **Use Case:** Represents a single embedding vector

---

#### 2D Array (Two-Dimensional)
- **Structure:** Multiple rows and columns (like a table/matrix)
- **Shape:** `(m, n)` where m = rows, n = columns
- **Example:**
```python
  array_2d = [[0.12, 0.56, 0.90, 0.34, 0.78],   # Row 0
              [0.15, 0.59, 0.91, 0.36, 0.75]]   # Row 1
  shape: (2, 5)  # 2 rows, 5 columns
```
- **Use Case:** Represents multiple embedding vectors (batch processing)

---

#### Accessing Elements

| Operation | Result | Type |
|-----------|--------|------|
| `array_2d[0]` | `[0.12, 0.56, 0.90, 0.34, 0.78]` | 1D array |
| `array_2d[1]` | `[0.15, 0.59, 0.91, 0.36, 0.75]` | 1D array |
| `array_2d[0, 2]` | `0.90` | Single value |

---

#### Why Shape is Always (1, 512) for CLIP-ViT-B-32?

##### Breaking Down (1, 512):
- **First dimension (1):** Number of texts you pass
  - You pass `[text]` → 1 text in the list → produces 1 row
  - If you passed `[text1, text2, text3]` → would be (3, 512)

- **Second dimension (512):** Embedding vector size of the model
  - `clip-ViT-B-32` always outputs **512-dimensional embeddings**
  - This is fixed by the model architecture
  - Every text gets converted to a vector of 512 numbers

##### Example:
```python
clip_model.encode([text])           # 1 text → shape: (1, 512)
clip_model.encode([text1, text2])   # 2 texts → shape: (2, 512)
clip_model.encode([text1, text2, text3, text4])  # 4 texts → shape: (4, 512)
```

The **512 never changes** (model-specific), but the **first number changes** based on how many texts you process.

---

#### Key Takeaway
**In the code:** `text_emb[0]` extracts the first row from a 2D array, converting it to a 1D embedding vector.
```python
text_emb = clip_model.encode([text])  # Returns 2D array: (1, 512)
                                      # 1 text, 512-dim embedding
embedding = text_emb[0]               # Extract row 0 → 1D array: (512,)
```


In [8]:
doc = fitz.open(PDF_PATH)

for i in doc:
    # Extract text from the page organized into structured blocks (paragraphs/sections)
    print(i.get_text("blocks"))

# Each block in the output list will look like a tuple with 7 items:(x0, y0, x1, y1, "text", block_no, block_type)
    # x0, y0, x1, y1: The exact boundary coordinates of the text box on the page.
    # "text": The actual string of text inside that specific paragraph or heading (with lines separated by \n).
    # block_no: The sequence number of the block on the page (starting from 0).
    # block_type: A number telling you what it is (0 means it is a text block, 1 means it is an image block)

[(124.31300354003906, 72.89631652832031, 487.8945007324219, 98.79853820800781, 'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\n', 0, 0), (267.593994140625, 100.79133605957031, 346.4983215332031, 112.74653625488281, 'scholarly works.\n', 1, 0), (211.48800659179688, 148.8298797607422, 399.89349365234375, 166.04527282714844, 'Attention Is All You Need\n', 2, 0), (132.9080047607422, 233.53939819335938, 203.8866729736328, 245.01112365722656, 'Ashish Vaswani∗\n', 3, 0), (116.68099975585938, 246.04847717285156, 216.05787658691406, 267.1680908203125, 'Google Brain\navaswani@google.com\n', 4, 0), (239.06199645996094, 233.53939819335938, 304.8396911621094, 245.01112365722656, 'Noam Shazeer∗\n', 5, 0), (230.69200134277344, 246.04847717285156, 309.1475830078125, 267.1680908203125, 'Google Brain\nnoam@google.com\n', 6, 0), (323.7870178222656, 233.53939819335938, 407.4730529785156, 267.168

In [9]:
doc
doc.metadata

{'format': 'PDF 1.5',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'creator': 'LaTeX with hyperref',
 'producer': 'pdfTeX-1.40.25',
 'creationDate': 'D:20240410211143Z',
 'modDate': 'D:20240410211143Z',
 'trapped': '',
 'encryption': None}

In [10]:
for i in doc:
    print(i.get_images()) # Get a list of metadata tuples for all images referenced on the current page

# According to the output, there are 3 images referenced in total across the entire document.Here is the breakdown based 
# on your output lines:
    # Page 3: Contains 1 image reference (Im1 at XREF 128)
    # Page 4: Contains 2 image references (Im2 at XREF 182, and Im3 at XREF 183)
    # All other pages: Contain [], meaning they have 0 image references.


[]
[]
[(128, 175, 1520, 2239, 8, 'DeviceRGB', '', 'Im1', 'FlateDecode')]
[(182, 200, 445, 884, 8, 'DeviceRGB', '', 'Im2', 'FlateDecode'), (183, 201, 835, 1282, 8, 'DeviceRGB', '', 'Im3', 'FlateDecode')]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]


### Strategy: Extracting Figures + Captions using Bounding Boxes

- *Bounding box:* an invisible rectangle drawn around something on the page, described by 4 coordinates — `(x0, y0)` = top-left corner, `(x1, y1)` = bottom-right corner. It's just numbers, not a visible drawing.
- For every image found on a page, `page.get_image_rects(xref)` gives us this box, so we know exactly where the image sits and — most importantly — where it **ends** (`rect.y1`, the bottom edge).

- *Caption heuristic:* the text paragraph that starts closest **below** the image's bottom edge is treated as its caption (e.g., "Figure 1: The Transformer - model architecture"). This works because papers almost always place captions directly under their figures.
    1. Collect all text blocks whose top edge (`y0`) is below the image's bottom edge (`rect.y1`).
    2. If none exist (e.g., image sits at the very bottom of the page), fall back to considering *all* text blocks on that page instead.
    3. Pick the text block whose top edge is **closest** to the image's bottom edge → treat it as the caption.

- Each image is then:
    - Extracted as raw bytes (`doc.extract_image(xref)`) and saved to disk in the `figures/` folder.
    - Recorded as one dictionary with: `id`, `page_number`, bounding box coordinates, `caption`, and `image_path`.

- These records (`image_records`) will later be turned into embeddings using the CLIP model and stored in ChromaDB's **image collection**, with the caption kept as searchable metadata alongside each image vector.

- *Note:* A similar (but simpler) pass is done separately for plain body text — looping over each page's text blocks, filtering out tiny fragments (< `MIN_CHARS`), and saving the rest (with the

### What is the below code trying to do ?

The below code uses PyMuPDF library to:

1. Go through every page in a PDF.
2. Find all images on each page.
3. Find the text blocks on that page.
4. Identify the text that is most likely an image caption.
5. Extract each image and save it to a folder.
6. Store information about each image in a Python list.

For every extracted image, the code is building something like below:
```python
{
    "id": "p2_img1",
    "page_number": 2,
    "bbox_x0": 72.0,
    "bbox_y0": 150.0,
    "bbox_x1": 500.0,
    "bbox_y1": 400.0,
    "caption": "Figure 1: Sales growth by region",
    "image_path": "figures/page_02_img_1.png"
}
```
This dictionary is the image's metadata. The actual image is saved separately as a file.

In [ ]:
# Create a list to store image metadata. Expected to contain a list -> Each item in the list is expected to be a dictionary ->
# Dictionary keys are expected to be strings -> Dictionary values can be of different data types.
# Example of what each record will look like:
#  [
#     {"id": "p1_img1", "page_number": 1},
#     {"id": "p2_img1", "page_number": 2}
#  ]

image_records: List[Dict[str, Any]] = []

print("Extracting images and captions...")


# Process the PDF one page at a time.
# start=1 keeps page numbers aligned with the way pages are numbered in the PDF.
for page_index, page in enumerate(doc, start=1):
    # Get text blocks along with their positions on the page.
    # A block typically contains: (x0, y0, x1, y1, text, ...).
    blocks = page.get_text("blocks")

    # PyMuPDF returns each text block as a tuple in the format:
    # (x0, y0, x1, y1, text, block_no, block_type).
    # The text content is stored at index 4, so b[4] gives us the actual text.
    # .strip() removes surrounding whitespace and helps us skip empty text blocks.
    # Example: (50, 100, 500, 130, "  Hello World\n", 0, 0) → b[4].strip() → "Hello World".
    # These are not useful when looking for a possible image caption.
    text_blocks = [b for b in blocks if b[4].strip()]

    # Get the images associated with the current page.
    images = page.get_images(full=True)
    print(f"Page {page_index}: found {len(images)} images")

    # Process each image separately.
    for img_index, img in enumerate(images, start=1):
        # The first value in the image tuple is the PDF object's xref.
        # This reference is used to locate and extract the image.
        xref = img[0]

        # Find the image's position on the page.
        # An image may have multiple placements, so PyMuPDF returns a list of rectangles.
        rects = page.get_image_rects(xref)
        if not rects:
            # Skip the image if its position cannot be determined.
            continue

        # Use the first placement when recording the image's position.
        rect = rects[0]

        # Look for text blocks whose top edge is at or below the image's bottom edge.
        # This assumes that an image caption is usually placed below the image.
        below_blocks = [b for b in text_blocks if b[1] >= rect.y1]

        # If no text is found below the image, fall back to all text blocks on the page.
        candidates = below_blocks if below_blocks else text_blocks

        if candidates:
            # Select the candidate whose top edge is closest to the image's bottom edge.
            # This is a simple proximity-based caption heuristic, not a guaranteed caption detector.
            closest_block = min(
                candidates,
                key=lambda b: abs(b[1] - rect.y1)
            )

            # Normalize the caption into a single line and remove surrounding whitespace.
            caption_text = closest_block[4].strip().replace("\n", " ")
        else:
            # Keep the field consistent when no possible caption is available.
            caption_text = ""

        # Extract the image's binary data and its file extension.
        img_data = doc.extract_image(xref)
        img_bytes = img_data["image"]
        img_ext = img_data.get("ext", "png")

        # Build a readable filename such as page_02_img_1.png.
        img_filename = f"page_{page_index:02d}_img_{img_index}.{img_ext}"
        img_path = os.path.join(FIGURES_DIR, img_filename)

        # Save the image bytes as a binary file.
        with open(img_path, "wb") as f:
            f.write(img_bytes)

        # Store the image's identifying information, page position, caption, and file path.
        image_records.append({
            "id": f"p{page_index}_img{img_index}",
            "page_number": page_index,
            "bbox_x0": float(rect.x0),
            "bbox_y0": float(rect.y0),
            "bbox_x1": float(rect.x1),
            "bbox_y1": float(rect.y1),
            "caption": caption_text,
            "image_path": img_path,
        })


print(f"Total extracted images: {len(image_records)}")

Extracting images and captions...
Page 1: found 0 images
Page 2: found 0 images
Page 3: found 1 images
Page 4: found 2 images
Page 5: found 0 images
Page 6: found 0 images
Page 7: found 0 images
Page 8: found 0 images
Page 9: found 0 images
Page 10: found 0 images
Page 11: found 0 images
Page 12: found 0 images
Page 13: found 0 images
Page 14: found 0 images
Page 15: found 0 images
Total extracted images: 3


In [12]:
text_records: List[Dict[str, Any]] = []

print("Extracting text chunks...")
MIN_CHARS = 80  # ignore very tiny text fragments

for page_index, page in enumerate(doc, start=1):
    blocks = page.get_text("blocks")  # (x0, y0, x1, y1, text, ...)
    for block_idx, b in enumerate(blocks):
        text = b[4].strip()
        if len(text) < MIN_CHARS:
            continue

        x0, y0, x1, y1 = b[0], b[1], b[2], b[3]

        text_records.append({
            "id": f"p{page_index}_block{block_idx}",
            "page_number": page_index,
            "bbox_x0": float(x0),
            "bbox_y0": float(y0),
            "bbox_x1": float(x1),
            "bbox_y1": float(y1),
            "text": text,
        })

print(f"Total extracted text chunks: {len(text_records)}")

Extracting text chunks...
Total extracted text chunks: 141


In [13]:
text_records

[{'id': 'p1_block0',
  'page_number': 1,
  'bbox_x0': 124.31300354003906,
  'bbox_y0': 72.89631652832031,
  'bbox_x1': 487.8945007324219,
  'bbox_y1': 98.79853820800781,
  'text': 'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or'},
 {'id': 'p1_block15',
  'page_number': 1,
  'bbox_x0': 143.5570068359375,
  'bbox_y0': 413.33428955078125,
  'bbox_x1': 469.78717041015625,
  'bbox_y1': 576.0990600585938,
  'text': 'The dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence and convolutions\nentirely. Experiments on two machine translation tasks show these models to\nbe supe

In [14]:
fig_collection = client.get_or_create_collection(
    name=FIG_COLLECTION_NAME,
    metadata={"type": "image", "source": "attention_pdf"},
)

text_collection = client.get_or_create_collection(
    name=TEXT_COLLECTION_NAME,
    metadata={"type": "text", "source": "attention_pdf"},
)


In [15]:
fig_ids = []
fig_embeddings = []
fig_metadatas = []
fig_documents = []

for rec in image_records:
    img_emb = embed_image(rec["image_path"])

    fig_ids.append(rec["id"])
    fig_embeddings.append(img_emb.tolist())
    fig_metadatas.append({
        "page_number": int(rec["page_number"]),
        "bbox_x0": rec["bbox_x0"],
        "bbox_y0": rec["bbox_y0"],
        "bbox_x1": rec["bbox_x1"],
        "bbox_y1": rec["bbox_y1"],
        "image_path": rec["image_path"],
        "caption": rec["caption"],
    })
    fig_documents.append(
        f"Image from 'Attention Is All You Need' on page {rec['page_number']}. "
        f"Caption: {rec['caption']}"
    )

if fig_ids:
    fig_collection.add(
        ids=fig_ids,
        embeddings=fig_embeddings,
        metadatas=fig_metadatas,
        documents=fig_documents,
    )

Batches: 100%|██████████| 1/1 [00:00<00:00, 13.06it/s]


In [16]:
txt_ids = []
txt_embeddings = []
txt_metadatas = []
txt_documents = []

for rec in text_records:
    emb = embed_text(rec["text"])

    txt_ids.append(rec["id"])
    txt_embeddings.append(emb.tolist())
    txt_metadatas.append({
        "page_number": int(rec["page_number"]),
        "bbox_x0": rec["bbox_x0"],
        "bbox_y0": rec["bbox_y0"],
        "bbox_x1": rec["bbox_x1"],
        "bbox_y1": rec["bbox_y1"],
    })
    txt_documents.append(rec["text"])

if txt_ids:
    text_collection.add(
        ids=txt_ids,
        embeddings=txt_embeddings,
        metadatas=txt_metadatas,
        documents=txt_documents,
    )

In [17]:
def answer_query(
    query: str,
    top_k_text: int = 5,
    top_k_img: int = 5,
    top_k_overall: int = 8,
) -> List[Dict[str, Any]]:
    """
    Search both text chunks and images in Chroma Cloud,
    merge results by distance, and return/print top hits.
    """
    q_emb = embed_text(query)

    combined: List[Dict[str, Any]] = []

    # --- Text search ---
    text_results = text_collection.query(
        query_embeddings=[q_emb.tolist()],
        n_results=top_k_text,
        include=["metadatas", "documents", "distances"],
    )

    for meta, doc, dist in zip(
        text_results["metadatas"][0],
        text_results["documents"][0],
        text_results["distances"][0],
    ):
        combined.append({
            "type": "text",
            "distance": float(dist),
            "page_number": meta["page_number"],
            "bbox": (
                meta["bbox_x0"],
                meta["bbox_y0"],
                meta["bbox_x1"],
                meta["bbox_y1"],
            ),
            "content": doc,
        })

    # --- Image search ---
    fig_results = fig_collection.query(
        query_embeddings=[q_emb.tolist()],
        n_results=top_k_img,
        include=["metadatas", "documents", "distances"],
    )

    for meta, doc, dist in zip(
        fig_results["metadatas"][0],
        fig_results["documents"][0],
        fig_results["distances"][0],
    ):
        combined.append({
            "type": "image",
            "distance": float(dist),
            "page_number": meta["page_number"],
            "image_path": meta["image_path"],
            "caption": meta.get("caption", ""),
            "content": doc,
        })

    # Sort by similarity (lower distance = more similar)
    combined.sort(key=lambda x: x["distance"])
    combined = combined[:top_k_overall]

    # Pretty-print
    print(f"\n=== Query: {query} ===")
    for rank, item in enumerate(combined, start=1):
        print(f"\nRank {rank} | type: {item['type']} | distance: {item['distance']:.4f}")
        print("  Page:", item["page_number"])
        if item["type"] == "text":
            preview = item["content"].replace("\n", " ")
            if len(preview) > 260:
                preview = preview[:260] + "..."
            print("  Text:", preview)
        else:
            print("  Image path:", item["image_path"])
            print("  Caption   :", item.get("caption", ""))

    return combined

In [18]:
answer_query("transfermer model architecture",top_k_text=2, top_k_img=1, top_k_overall=3)


=== Query: transfermer model architecture ===

Rank 1 | type: text | distance: 32.6148
  Page: 10
  Text: For translation tasks, the Transformer can be trained significantly faster than architectures based on recurrent or convolutional layers. On both WMT 2014 English-to-German and WMT 2014 English-to-French translation tasks, we achieve a new state of the art. In...

Rank 2 | type: text | distance: 35.0930
  Page: 2
  Text: In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and ca...

Rank 3 | type: image | distance: 138.0081
  Page: 4
  Image path: c:\Users\AdityaRajPanda\OneDrive - Cubastion Consulting Pvt Ltd\Desktop\genai-bootcamp\03_VectorDBs\figures\page_04_img_2.png
  Caption   : Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several att

[{'type': 'text',
  'distance': 32.61478,
  'page_number': 10,
  'bbox': (108.0, 427.23828125, 503.99761962890625, 470.0030822753906),
  'content': 'For translation tasks, the Transformer can be trained significantly faster than architectures based\non recurrent or convolutional layers. On both WMT 2014 English-to-German and WMT 2014\nEnglish-to-French translation tasks, we achieve a new state of the art. In the former task our best\nmodel outperforms even all previously reported ensembles.'},
 {'type': 'text',
  'distance': 35.09295,
  'page_number': 2,
  'bbox': (107.69100189208984,
   300.9082946777344,
   505.73870849609375,
   343.6730651855469),
  'content': 'In this work we propose the Transformer, a model architecture eschewing recurrence and instead\nrelying entirely on an attention mechanism to draw global dependencies between input and output.\nThe Transformer allows for significantly more parallelization and can reach a new state of the art in\ntranslation quality after bei

In [20]:
answer_query("Explain me attention",top_k_text=2, top_k_img=1, top_k_overall=3)


=== Query: Explain me attention ===

Rank 1 | type: text | distance: 41.8693
  Page: 1
  Text: Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or

Rank 2 | type: text | distance: 44.4472
  Page: 2
  Text: Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading...

Rank 3 | type: image | distance: 192.5309
  Page: 4
  Image path: c:\Users\AdityaRajPanda\OneDrive - Cubastion Consulting Pvt Ltd\Desktop\genai-bootcamp\03_VectorDBs\figures\page_04_img_2.png
  Caption   : Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several attention layers running in parallel.


[{'type': 'text',
  'distance': 41.869343,
  'page_number': 1,
  'bbox': (124.31300354003906,
   72.89631652832031,
   487.8945007324219,
   98.7985382080078),
  'content': 'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or'},
 {'type': 'text',
  'distance': 44.44717,
  'page_number': 2,
  'bbox': (108.0, 493.1185607910156, 505.2498779296875, 535.7550659179688),
  'content': 'Self-attention, sometimes called intra-attention is an attention mechanism relating different positions\nof a single sequence in order to compute a representation of the sequence. Self-attention has been\nused successfully in a variety of tasks including reading comprehension, abstractive summarization,\ntextual entailment and learning task-independent sentence representations [4, 27, 28, 22].'},
 {'type': 'image',
  'distance': 192.53087,
  'page_number': 4,
  'image_path': 'c:\\Users\\AdityaRajPanda\\OneDr